## 13. Spatial Entropy of Codebook Assignments

For each spatial location in the quantized feature map, compute the entropy of the codebook index distribution across all subjects. Low entropy means a location is "committed" to a specific codebook vector (always picks the same code), while high entropy means the location varies across subjects.

This reveals whether the VQ-VAE learns spatially rigid templates or flexible, subject-dependent encodings.

In [ ]:
# ── Collect spatial codebook indices across all subjects ──
# spatial_indices[lvl] will be an array of shape (N, D, H, W) with the codebook
# index assigned at each spatial location for each subject.

spatial_indices = [[] for _ in range(nb_levels)]

with torch.no_grad():
    for batch in tqdm(loader, desc="Collecting spatial indices"):
        images = batch["image"].to(DEVICE)
        _, _, _, _, id_outputs, _ = model(images, return_recon=True, pool_only=True)
        # Reverse to finest-first ordering (same convention as cell above)
        id_outputs = id_outputs[::-1]

        for lvl in range(nb_levels):
            spatial_indices[lvl].append(id_outputs[lvl].cpu().numpy())

for lvl in range(nb_levels):
    spatial_indices[lvl] = np.concatenate(spatial_indices[lvl], axis=0)  # (N, D, H, W)
    print(f"Level {lvl}: spatial indices shape {spatial_indices[lvl].shape}")

In [ ]:
# ── Compute per-location entropy ──
from scipy.stats import entropy as sp_entropy

spatial_entropy = []  # one per level

for lvl in range(nb_levels):
    indices = spatial_indices[lvl]  # (N, D, H, W)
    N_subj = indices.shape[0]
    spatial_shape = indices.shape[1:]  # (D, H, W)

    # Compute histogram over codebook entries at each spatial location
    flat = indices.reshape(N_subj, -1)  # (N, D*H*W)
    n_locations = flat.shape[1]

    ent_map = np.zeros(n_locations)
    for loc in range(n_locations):
        counts = np.bincount(flat[:, loc].astype(int), minlength=nb_entries)
        probs = counts / counts.sum()
        ent_map[loc] = sp_entropy(probs, base=2)  # bits

    ent_map = ent_map.reshape(spatial_shape)
    spatial_entropy.append(ent_map)

    max_entropy = np.log2(nb_entries)
    print(f"Level {lvl}: entropy range [{ent_map.min():.3f}, {ent_map.max():.3f}] bits "
          f"(max possible = {max_entropy:.2f} bits for {nb_entries} codes)")
    print(f"  Mean entropy: {ent_map.mean():.3f} bits, "
          f"Fraction of locations with entropy < 1 bit: {(ent_map < 1.0).mean():.1%}")

In [ ]:
# ── Visualize spatial entropy maps (axial mid-slices) ──

for lvl in range(nb_levels):
    ent = spatial_entropy[lvl]  # (D, H, W)
    D, H, W = ent.shape
    max_ent = np.log2(nb_entries)

    # Show 5 evenly spaced axial slices
    slice_idxs = np.linspace(0, D - 1, 5, dtype=int)

    fig, axes = plt.subplots(1, len(slice_idxs), figsize=(4 * len(slice_idxs), 4))
    fig.suptitle(f"Level {lvl} — Spatial Entropy (bits) of Codebook Assignments", fontsize=14)

    for ax, s in zip(axes, slice_idxs):
        im = ax.imshow(ent[s], cmap="hot", vmin=0, vmax=max_ent, origin="lower")
        ax.set_title(f"Slice {s}")
        ax.axis("off")

    fig.colorbar(im, ax=axes, shrink=0.8, label="Entropy (bits)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Entropy histogram: distribution of commitment across locations ──

fig, axes = plt.subplots(1, nb_levels, figsize=(6 * nb_levels, 4))
if nb_levels == 1:
    axes = [axes]

for lvl, ax in enumerate(axes):
    ent = spatial_entropy[lvl].ravel()
    max_ent = np.log2(nb_entries)
    ax.hist(ent, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(ent.mean(), color="red", linestyle="--", label=f"Mean = {ent.mean():.2f}")
    ax.axvline(max_ent, color="gray", linestyle=":", label=f"Max possible = {max_ent:.2f}")
    ax.set_xlabel("Entropy (bits)")
    ax.set_ylabel("Number of spatial locations")
    ax.set_title(f"Level {lvl}")
    ax.legend(fontsize=9)

fig.suptitle("Distribution of Per-Location Entropy", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Most and least committed locations: show dominant code frequency ──

for lvl in range(nb_levels):
    ent = spatial_entropy[lvl]  # (D, H, W)
    indices = spatial_indices[lvl]  # (N, D, H, W)
    N_subj = indices.shape[0]

    # For each location, find the most frequent code and its frequency
    flat = indices.reshape(N_subj, -1)
    n_locations = flat.shape[1]
    dominant_freq = np.zeros(n_locations)
    dominant_code = np.zeros(n_locations, dtype=int)

    for loc in range(n_locations):
        counts = np.bincount(flat[:, loc].astype(int), minlength=nb_entries)
        dominant_code[loc] = counts.argmax()
        dominant_freq[loc] = counts.max() / counts.sum()

    dominant_freq = dominant_freq.reshape(ent.shape)
    dominant_code = dominant_code.reshape(ent.shape)

    D, H, W = ent.shape
    mid = D // 2

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Mid-axial Slice {mid}", fontsize=14)

    im0 = axes[0].imshow(ent[mid], cmap="hot", vmin=0, vmax=np.log2(nb_entries), origin="lower")
    axes[0].set_title("Entropy (bits)")
    fig.colorbar(im0, ax=axes[0], shrink=0.8)

    im1 = axes[1].imshow(dominant_freq[mid], cmap="RdYlGn", vmin=0, vmax=1, origin="lower")
    axes[1].set_title("Dominant Code Frequency")
    fig.colorbar(im1, ax=axes[1], shrink=0.8)

    im2 = axes[2].imshow(dominant_code[mid], cmap="tab20", origin="lower")
    axes[2].set_title("Dominant Code Index")
    fig.colorbar(im2, ax=axes[2], shrink=0.8)

    for ax in axes:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    # Print summary
    print(f"  Fully committed (>95% same code): {(dominant_freq > 0.95).mean():.1%} of locations")
    print(f"  Highly variable (<50% dominant):  {(dominant_freq < 0.50).mean():.1%} of locations")

## 13b. Regional Entropy of Codebook Assignments

Per-voxel entropy assumes perfect spatial alignment in code-space, but downsampling and large receptive fields mean a single-voxel shift can change which code is assigned. Computing entropy over local regions (patches of size `n×n×n`) pools nearby assignments together, smoothing out these small misalignments and revealing whether a *region* consistently uses certain codes across subjects.

In [ ]:
# ── Regional entropy: pool codebook indices over n×n×n patches ──

def regional_entropy(indices, region_size, nb_entries):
    """Compute entropy over non-overlapping spatial regions.

    Args:
        indices: (N_subjects, D, H, W) array of codebook indices.
        region_size: int or tuple (rD, rH, rW) — patch size for pooling.
        nb_entries: number of codebook entries (for bincount).

    Returns:
        ent_map: (D', H', W') array of entropy values per region.
        region_grid: tuple of (rD, rH, rW) used.
    """
    if isinstance(region_size, int):
        region_size = (region_size, region_size, region_size)
    rD, rH, rW = region_size
    N, D, H, W = indices.shape

    # Number of full regions along each axis
    nD, nH, nW = D // rD, H // rH, W // rW

    # Trim to exact multiples
    trimmed = indices[:, :nD * rD, :nH * rH, :nW * rW]

    # Reshape into (N, nD, rD, nH, rH, nW, rW) then merge subject + patch dims
    blocks = trimmed.reshape(N, nD, rD, nH, rH, nW, rW)
    # Move region axes together: (nD, nH, nW, N*rD*rH*rW)
    blocks = blocks.transpose(1, 3, 5, 0, 2, 4, 6).reshape(nD, nH, nW, -1)

    ent_map = np.zeros((nD, nH, nW))
    for di in range(nD):
        for hi in range(nH):
            for wi in range(nW):
                counts = np.bincount(blocks[di, hi, wi].astype(int), minlength=nb_entries)
                probs = counts / counts.sum()
                ent_map[di, hi, wi] = sp_entropy(probs, base=2)

    return ent_map, region_size

In [ ]:
# ── Compare regional entropy at different scales ──

REGION_SIZES = [2, 3, 4]  # n×n×n patch sizes to compare

for lvl in range(nb_levels):
    indices = spatial_indices[lvl]  # (N, D, H, W)
    max_ent = np.log2(nb_entries)

    fig, axes = plt.subplots(1, len(REGION_SIZES) + 1, figsize=(5 * (len(REGION_SIZES) + 1), 4))
    fig.suptitle(f"Level {lvl} — Regional Entropy at Different Scales (mid-axial slice)", fontsize=14)

    # Per-voxel reference
    ent_voxel = spatial_entropy[lvl]
    mid = ent_voxel.shape[0] // 2
    im = axes[0].imshow(ent_voxel[mid], cmap="hot", vmin=0, vmax=max_ent, origin="lower")
    axes[0].set_title(f"Per-voxel (1×1×1)")
    axes[0].axis("off")

    for i, rs in enumerate(REGION_SIZES):
        ent_reg, used_rs = regional_entropy(indices, rs, nb_entries)
        mid_r = ent_reg.shape[0] // 2
        im = axes[i + 1].imshow(ent_reg[mid_r], cmap="hot", vmin=0, vmax=max_ent, origin="lower")
        axes[i + 1].set_title(f"Region {rs}×{rs}×{rs}")
        axes[i + 1].axis("off")

        print(f"Level {lvl}, region {rs}×{rs}×{rs}: "
              f"entropy range [{ent_reg.min():.3f}, {ent_reg.max():.3f}], "
              f"mean {ent_reg.mean():.3f} bits, "
              f"grid shape {ent_reg.shape}")

    fig.colorbar(im, ax=axes, shrink=0.8, label="Entropy (bits)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Entropy histogram comparison: per-voxel vs regional ──

for lvl in range(nb_levels):
    indices = spatial_indices[lvl]
    max_ent = np.log2(nb_entries)

    fig, axes = plt.subplots(1, len(REGION_SIZES) + 1, figsize=(5 * (len(REGION_SIZES) + 1), 4))
    fig.suptitle(f"Level {lvl} — Entropy Distribution: Per-Voxel vs Regional", fontsize=14)

    # Per-voxel
    ent_voxel = spatial_entropy[lvl].ravel()
    axes[0].hist(ent_voxel, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
    axes[0].axvline(ent_voxel.mean(), color="red", linestyle="--", label=f"Mean={ent_voxel.mean():.2f}")
    axes[0].set_title("Per-voxel")
    axes[0].set_xlabel("Entropy (bits)")
    axes[0].set_ylabel("Count")
    axes[0].legend(fontsize=8)

    for i, rs in enumerate(REGION_SIZES):
        ent_reg, _ = regional_entropy(indices, rs, nb_entries)
        ent_flat = ent_reg.ravel()
        axes[i + 1].hist(ent_flat, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
        axes[i + 1].axvline(ent_flat.mean(), color="red", linestyle="--", label=f"Mean={ent_flat.mean():.2f}")
        axes[i + 1].set_title(f"Region {rs}×{rs}×{rs}")
        axes[i + 1].set_xlabel("Entropy (bits)")
        axes[i + 1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

# VQ-VAE Codebook & Class Analysis

Analyzes how codebook usage patterns correlate with diagnostic classes (AD, CN, MCI).
Includes PCA/t-SNE visualizations of both codebook histograms and continuous encoder features.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency
from tqdm.auto import tqdm

from eval import load_model_from_checkpoint, get_transforms
from utils import load_items
from monai.data import Dataset

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Configuration

Set paths below before running.

In [ ]:
CHECKPOINT_PATH = "/home/ng24/projects/vqvae-smm/results/checkpoint_best.pt"  # TODO: fill in
CSV_PATH = "/home/ng24/projects/nmpevqvae/labels_cleaned_3class.csv"
DATAROOT = "/data/natalia/ADNI_registered"

SPACING = 2.0        # voxel spacing in mm (2.0 = faster, 1.0 = full res)
DOWNSAMPLE = 1.0     # extra downsampling factor applied AFTER resampling to SPACING
                     # e.g. 0.5 = halve each spatial dim, 1.0 = no change
CROP_MARGIN = 0      # voxels to crop from each edge of every spatial dim (0 = no crop)
BATCH_SIZE = 4
NUM_WORKERS = 4
MAX_SUBJECTS = None  # set to e.g. 100 to cap dataset size (None = use all)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CLASS_NAMES = ["AD", "CN", "MCI"]  # sorted order matches label_map in load_items
CLASS_COLORS = {"AD": "#e74c3c", "CN": "#2ecc71", "MCI": "#3498db"}

print(f"Device: {DEVICE}")

## 2. Load Model & Data

In [ ]:
model = load_model_from_checkpoint(CHECKPOINT_PATH, device=DEVICE)
model.eval()

nb_levels = model.nb_levels
nb_entries = model.codebooks[0].n_embed
print(f"Model: {nb_levels} levels, {nb_entries} codebook entries each")

In [ ]:
from monai.transforms import (
    Compose, LoadImaged, Lambdad, EnsureChannelFirstd,
    Spacingd, Orientationd, NormalizeIntensityd,
    ResizeWithPadOrCropd, Resized, CenterSpatialCropd, ToTensord,
)
from utils import _ensure_3d_image

items = load_items(DATAROOT, CSV_PATH)

# Optionally subsample to avoid OOM
if MAX_SUBJECTS is not None and len(items) > MAX_SUBJECTS:
    rng = np.random.RandomState(42)
    idx = rng.choice(len(items), MAX_SUBJECTS, replace=False)
    idx.sort()
    items = [items[i] for i in idx]

print(f"Using {len(items)} subjects")

# ── Step 1: probe spatial size from the first image ──────────────────────
# Load + resample + orient ONE image to discover the native spatial dims
# at the requested voxel spacing (no hardcoded sizes).
probe_transforms = [
    LoadImaged(keys=["image"], image_only=True),
    Lambdad(keys=["image"], func=_ensure_3d_image),
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
]
if SPACING != 1.0:
    probe_transforms.append(
        Spacingd(keys=["image"], pixdim=(SPACING, SPACING, SPACING), mode="bilinear")
    )
probe_transforms.append(Orientationd(keys=["image"], axcodes="RAS"))
probe_result = Compose(probe_transforms)({"image": items[0]["image"]})
native_size = tuple(probe_result["image"].shape[1:])  # (D, H, W) after channel dim
print(f"Native spatial size at {SPACING}mm spacing: {native_size}")

# ── Step 2: apply crop margin ────────────────────────────────────────────
if CROP_MARGIN > 0:
    spatial_size = tuple(s - 2 * CROP_MARGIN for s in native_size)
    assert all(s > 0 for s in spatial_size), (
        f"CROP_MARGIN={CROP_MARGIN} is too large for native size {native_size}"
    )
    print(f"After cropping {CROP_MARGIN}px per edge: {spatial_size}")
else:
    spatial_size = native_size

# ── Step 3: optional downsampling ────────────────────────────────────────
if DOWNSAMPLE < 1.0:
    spatial_size = tuple(max(1, int(s * DOWNSAMPLE)) for s in spatial_size)
    print(f"After downsampling (factor {DOWNSAMPLE}): {spatial_size}")

# ── Step 4: build the full transform pipeline ────────────────────────────
transform_list = [
    LoadImaged(keys=["image"], image_only=True),
    Lambdad(keys=["image"], func=_ensure_3d_image),
    EnsureChannelFirstd(keys=["image"], channel_dim="no_channel"),
]
if SPACING != 1.0:
    transform_list.append(
        Spacingd(keys=["image"], pixdim=(SPACING, SPACING, SPACING), mode="bilinear")
    )
transform_list.append(Orientationd(keys=["image"], axcodes="RAS"))

# Crop first (removes background edges), then downsample if requested
if CROP_MARGIN > 0:
    cropped_size = tuple(s - 2 * CROP_MARGIN for s in native_size)
    transform_list.append(
        CenterSpatialCropd(keys=["image"], roi_size=cropped_size)
    )

if DOWNSAMPLE < 1.0:
    # Resize to the downsampled target (trilinear interpolation)
    transform_list.append(
        Resized(keys=["image"], spatial_size=spatial_size, mode="trilinear")
    )
else:
    # Pad/crop to uniform size (handles minor per-subject size variations)
    transform_list.append(
        ResizeWithPadOrCropd(keys=["image"], spatial_size=spatial_size)
    )

transform_list.extend([
    NormalizeIntensityd(keys=["image"], nonzero=True, channel_wise=True),
    ToTensord(keys=["image"], track_meta=False),
])
transforms = Compose(transform_list)

dataset = Dataset(
    data=[{"image": it["image"]} for it in items],
    transform=transforms,
)

labels = np.array([it["label"] for it in items])
subjects = [it["subject"] for it in items]

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Final spatial size: {spatial_size}")
print(f"Class distribution: {dict(zip(CLASS_NAMES, np.bincount(labels)))}")

## 3. Extract Codebook Indices & Continuous Features

In [ ]:
# Storage: per-level lists (level 0 = finest, level nb_levels-1 = coarsest)
all_histograms = [[] for _ in range(nb_levels)]   # codebook usage histograms
all_pooled = [[] for _ in range(nb_levels)]        # continuous pooled encoder features

with torch.no_grad():
    for batch in tqdm(loader, desc="Extracting features"):
        images = batch["image"].to(DEVICE)

        # Single forward pass: return_recon=True ensures correct decoder conditioning
        # for all codebook levels (without it, non-coarsest levels get zero-padded
        # conditioning and produce wrong indices). pool_only=True returns pooled
        # encoder features as (B, C) vectors instead of full spatial maps.
        _, diffs, encoder_pools, _, id_outputs, _ = model(images, return_recon=True, pool_only=True)

        # CRITICAL: id_outputs is coarsest-first (appended during the
        # range(nb_levels-1, ..., -1) loop), but encoder_pools is finest-first
        # (appended during sequential encoder pass). Reverse id_outputs so both
        # use the same ordering: index 0 = finest, index nb_levels-1 = coarsest.
        id_outputs = id_outputs[::-1]

        B = images.shape[0]
        for lvl in range(nb_levels):
            # Codebook index histograms
            ids = id_outputs[lvl]  # (B, D, H, W)
            for b in range(B):
                hist = torch.bincount(ids[b].reshape(-1), minlength=nb_entries)
                hist = hist.float() / hist.sum()  # normalize to frequency
                all_histograms[lvl].append(hist.cpu().numpy())

            # Pooled encoder features
            all_pooled[lvl].append(encoder_pools[lvl].cpu().numpy())

# Stack into arrays
for lvl in range(nb_levels):
    all_histograms[lvl] = np.array(all_histograms[lvl])  # (N, nb_entries)
    all_pooled[lvl] = np.concatenate(all_pooled[lvl], axis=0)  # (N, C)

N = all_histograms[0].shape[0]
print(f"Extracted features for {N} subjects")
print(f"Level ordering: 0 = finest resolution, {nb_levels-1} = coarsest resolution\n")
for lvl in range(nb_levels):
    used = (all_histograms[lvl].sum(axis=0) > 0).sum()
    print(f"  Level {lvl}: histograms {all_histograms[lvl].shape}, pooled {all_pooled[lvl].shape}, "
          f"codes used across dataset: {used}/{nb_entries}")

## 4. Codebook Usage by Class

In [ ]:
for lvl in range(nb_levels):
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Codebook Usage by Class", fontsize=14)

    # --- Mean usage histogram per class ---
    ax = axes[0]
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        ax.bar(
            np.arange(nb_entries), mean_usage, alpha=0.5,
            label=cls_name, color=CLASS_COLORS[cls_name],
        )
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook usage")
    ax.legend()

    # --- Heatmap: classes x entries ---
    ax = axes[1]
    usage_matrix = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        usage_matrix[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)
    sns.heatmap(
        usage_matrix, ax=ax, cmap="viridis",
        yticklabels=CLASS_NAMES, xticklabels=False,
    )
    ax.set_xlabel("Codebook entry")
    ax.set_title("Usage heatmap (class x entry)")

    plt.tight_layout()
    plt.show()

In [ ]:
TOP_N = 10  # number of top codes to show per class

for lvl in range(nb_levels):
    print(f"{'=' * 70}")
    print(f"LEVEL {lvl}")
    print(f"{'=' * 70}")

    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_usage = all_histograms[lvl][mask].mean(axis=0)
        top_indices = np.argsort(mean_usage)[::-1][:TOP_N]

        print(f"\n  {cls_name} (n={mask.sum()}) — top {TOP_N} most-used codes:")
        print(f"  {'Code':>6s}  {'Freq':>8s}  {'Bar'}")
        print(f"  {'─' * 6}  {'─' * 8}  {'─' * 30}")
        for idx in top_indices:
            freq = mean_usage[idx]
            bar = "█" * int(freq * 500)  # scale for display
            print(f"  {idx:6d}  {freq:8.4f}  {bar}")

    # Also show codes with biggest difference between classes
    usage_per_class = np.zeros((len(CLASS_NAMES), nb_entries))
    for cls_idx in range(len(CLASS_NAMES)):
        usage_per_class[cls_idx] = all_histograms[lvl][labels == cls_idx].mean(axis=0)

    # Which class uses each code the most?
    dominant_class = np.argmax(usage_per_class, axis=0)
    max_diff = usage_per_class.max(axis=0) - usage_per_class.min(axis=0)
    top_diff = np.argsort(max_diff)[::-1][:TOP_N]

    print(f"\n  Codes with largest cross-class difference:")
    print(f"  {'Code':>6s}  {'Dominant':>8s}  {'MaxFreq':>8s}  {'MinFreq':>8s}  {'Diff':>8s}")
    print(f"  {'─' * 6}  {'─' * 8}  {'─' * 8}  {'─' * 8}  {'─' * 8}")
    for idx in top_diff:
        dom = CLASS_NAMES[dominant_class[idx]]
        mx = usage_per_class[:, idx].max()
        mn = usage_per_class[:, idx].min()
        print(f"  {idx:6d}  {dom:>8s}  {mx:8.4f}  {mn:8.4f}  {mx - mn:8.4f}")
    print()

## 5. Most Discriminative Codes (Chi-squared)

In [ ]:
TOP_K = 20

for lvl in range(nb_levels):
    chi2_scores = np.zeros(nb_entries)
    for code_idx in range(nb_entries):
        # Bin usage into quartiles for chi-squared
        usage = all_histograms[lvl][:, code_idx]
        bins = np.quantile(usage[usage > 0], [0.33, 0.66]) if (usage > 0).sum() > 10 else None
        if bins is None or len(np.unique(bins)) < 2:
            continue
        digitized = np.digitize(usage, bins)
        contingency = pd.crosstab(digitized, labels)
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, _, _ = chi2_contingency(contingency)
            chi2_scores[code_idx] = chi2

    top_codes = np.argsort(chi2_scores)[::-1][:TOP_K]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(range(TOP_K), chi2_scores[top_codes], color="steelblue")
    ax.set_xticks(range(TOP_K))
    ax.set_xticklabels(top_codes, rotation=45)
    ax.set_xlabel("Codebook entry index")
    ax.set_ylabel("Chi-squared statistic")
    ax.set_title(f"Level {lvl} — Top {TOP_K} most class-discriminative codes")
    plt.tight_layout()
    plt.show()

## 6. Mutual Information: Codebook Entry vs Class

In [ ]:
for lvl in range(nb_levels):
    mi_scores = mutual_info_classif(
        all_histograms[lvl], labels, discrete_features=False, random_state=42,
    )

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(np.arange(nb_entries), mi_scores, color="darkorange", width=1.0)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mutual information (nats)")
    ax.set_title(f"Level {lvl} — MI between codebook entry usage and class label")
    plt.tight_layout()
    plt.show()

    top10 = np.argsort(mi_scores)[::-1][:10]
    print(f"Level {lvl} top-10 MI codes: {top10.tolist()}")
    print(f"  MI values: {mi_scores[top10].round(4).tolist()}")

## 7. PCA & t-SNE of Codebook Usage Histograms

In [ ]:
def scatter_by_class(ax, coords, labels, class_names, class_colors, title):
    for cls_idx, cls_name in enumerate(class_names):
        mask = labels == cls_idx
        ax.scatter(
            coords[mask, 0], coords[mask, 1],
            c=class_colors[cls_name], label=cls_name,
            alpha=0.6, s=15, edgecolors="none",
        )
    ax.legend(markerscale=2)
    ax.set_title(title)


for lvl in range(nb_levels):
    X = all_histograms[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Codebook Histogram Embeddings", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 8. PCA & t-SNE of Continuous Encoder Features

In [ ]:
for lvl in range(nb_levels):
    X = all_pooled[lvl]

    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"Level {lvl} — Continuous Encoder Features", fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    axes[0].set_xlabel("PC1")
    axes[0].set_ylabel("PC2")

    scatter_by_class(
        axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE",
    )
    axes[1].set_xlabel("t-SNE 1")
    axes[1].set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.show()

## 9. Combined Multi-Level Feature Analysis

Concatenate features across all levels for a joint view.

In [ ]:
# Concatenate histograms across levels
X_hist_all = np.concatenate(all_histograms, axis=1)
X_pool_all = np.concatenate(all_pooled, axis=1)

for name, X in [("Codebook histograms (all levels)", X_hist_all),
                ("Pooled features (all levels)", X_pool_all)]:
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)
    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    X_tsne = tsne.fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(name, fontsize=14)

    scatter_by_class(
        axes[0], X_pca, labels, CLASS_NAMES, CLASS_COLORS,
        f"PCA (var explained: {pca.explained_variance_ratio_.sum():.1%})",
    )
    scatter_by_class(axes[1], X_tsne, labels, CLASS_NAMES, CLASS_COLORS, "t-SNE")

    plt.tight_layout()
    plt.show()

## 10. Codebook Vector Replacement & Reconstruction

Replace a specific codebook entry at a given level with another entry, decode
back to an image, and compare with the original reconstruction.

In [ ]:
import torch.nn.functional as F


@torch.no_grad()
def decode_from_indices(model, code_indices):
    """Decode a list of codebook index tensors back to an image (3D-safe).

    Args:
        model: VQVAE model.
        code_indices: list of LongTensors, one per level.
            Level ordering must match model convention (level 0 = finest).
            Each tensor has shape (B, D_l, H_l, W_l).

    Returns:
        Reconstructed image tensor (B, C, D, H, W).
    """
    decoder_outputs = []
    code_outputs = []
    upscale_counts = []

    for l in range(model.nb_levels - 1, -1, -1):
        codebook = model.codebooks[l]
        decoder = model.decoders[l]

        # embed_code -> (B, D, H, W, embed_dim), permute to (B, embed_dim, D, H, W)
        code_q = codebook.embed_code(code_indices[l]).permute(0, 4, 1, 2, 3)

        # Upscale previous code outputs
        upscaled_codes = []
        target_size = code_q.shape[2:]
        for i, c in enumerate(code_outputs):
            upscaled = model.upscalers[i](c, upscale_counts[i])
            if upscaled.shape[2:] != target_size:
                upscaled = F.interpolate(
                    upscaled, size=target_size, mode="trilinear", align_corners=False,
                )
            upscaled_codes.append(upscaled)
        code_outputs = upscaled_codes
        upscale_counts = [u + 1 for u in upscale_counts]

        decoder_in = torch.cat([code_q, *code_outputs], dim=1)
        decoder_outputs.append(decoder(decoder_in))

        code_outputs.append(code_q)
        upscale_counts.append(0)

    return decoder_outputs[-1]

In [ ]:
# ── Configuration ────────────────────────────────────────────────
SUBJECT_IDX = 0            # index into `items` list
TARGET_LEVEL = 2           # codebook level to modify (0=finest, nb_levels-1=coarsest)
OLD_CODE = 10              # codebook entry index to replace
NEW_CODE = 50              # replacement codebook entry index
SLICE_AXIS = 2             # 0=sagittal, 1=coronal, 2=axial

# ── Encode the subject ──────────────────────────────────────────
sample = dataset[SUBJECT_IDX]
image = sample["image"].unsqueeze(0).to(DEVICE)  # (1, C, D, H, W)

with torch.no_grad():
    recon_orig, _, _, _, id_outputs_raw = model(image, return_recon=True)

# id_outputs from the forward pass is coarsest-first (appended during the
# range(nb_levels-1, ..., -1) loop).  Reverse so index 0 = level 0 (finest).
id_outputs = id_outputs_raw[::-1]

# ── Diagnostics: how different are the two code embeddings? ─────
codebook = model.codebooks[TARGET_LEVEL]
emb_old = codebook.embed[:, OLD_CODE]   # (embed_dim,)
emb_new = codebook.embed[:, NEW_CODE]   # (embed_dim,)
l2_dist = (emb_old - emb_new).norm().item()
cos_sim = F.cosine_similarity(emb_old.unsqueeze(0), emb_new.unsqueeze(0)).item()
print(f"Embedding comparison (code {OLD_CODE} vs {NEW_CODE}):")
print(f"  L2 distance:       {l2_dist:.4f}")
print(f"  Cosine similarity: {cos_sim:.4f}")

# For context, show distribution of all pairwise distances
all_embeds = codebook.embed.T  # (nb_entries, embed_dim)
sample_idx = torch.randint(0, all_embeds.shape[0], (200,))
sample_embeds = all_embeds[sample_idx]
pairwise = torch.cdist(sample_embeds.unsqueeze(0), sample_embeds.unsqueeze(0)).squeeze()
mask = torch.triu(torch.ones_like(pairwise, dtype=torch.bool), diagonal=1)
print(f"  Pairwise L2 stats (sample of 200 codes): "
      f"mean={pairwise[mask].mean():.4f}, std={pairwise[mask].std():.4f}, "
      f"min={pairwise[mask].min():.4f}, max={pairwise[mask].max():.4f}")

# ── Replace codes ───────────────────────────────────────────────
modified_ids = [ids.clone() for ids in id_outputs]
n_replaced = (modified_ids[TARGET_LEVEL] == OLD_CODE).sum().item()
modified_ids[TARGET_LEVEL][modified_ids[TARGET_LEVEL] == OLD_CODE] = NEW_CODE
print(f"\nLevel {TARGET_LEVEL}: replaced {n_replaced} voxels from code {OLD_CODE} → {NEW_CODE}")
print(f"  Code map shapes: {[tuple(ids.shape) for ids in id_outputs]}")

if n_replaced == 0:
    print("⚠️  No voxels to replace! Try a different OLD_CODE that actually appears in this subject's code map.")
    # Show which codes are actually used at this level
    unique_codes = id_outputs[TARGET_LEVEL].unique().cpu().numpy()
    print(f"  Codes present at level {TARGET_LEVEL}: {sorted(unique_codes.tolist())}")

# ── Decode original & modified ──────────────────────────────────
with torch.no_grad():
    recon_orig_from_codes = decode_from_indices(model, id_outputs)
    recon_modified = decode_from_indices(model, modified_ids)

# Interpolate to match input spatial size if needed
input_shape = image.shape[2:]
if recon_orig_from_codes.shape[2:] != input_shape:
    recon_orig_from_codes = F.interpolate(
        recon_orig_from_codes, size=input_shape, mode="trilinear", align_corners=False,
    )
if recon_modified.shape[2:] != input_shape:
    recon_modified = F.interpolate(
        recon_modified, size=input_shape, mode="trilinear", align_corners=False,
    )

# ── Difference statistics ───────────────────────────────────────
diff = (recon_modified - recon_orig_from_codes).abs()
print(f"\nReconstruction difference stats:")
print(f"  min={diff.min().item():.6f}, max={diff.max().item():.6f}, "
      f"mean={diff.mean().item():.6f}, std={diff.std().item():.6f}")
print(f"  Original recon range: [{recon_orig_from_codes.min().item():.4f}, {recon_orig_from_codes.max().item():.4f}]")
print(f"  Relative max diff:    {diff.max().item() / (recon_orig_from_codes.abs().max().item() + 1e-8):.4%}")

# ── Visualize ───────────────────────────────────────────────────
def get_mid_slice(vol, axis):
    """Get the middle slice and quarter slices along a given axis from a (C, D, H, W) tensor."""
    idx_mid = vol.shape[axis + 1] // 2  # +1 to skip channel dim
    idx_q1 = vol.shape[axis + 1] // 4
    idx_q3 = 3 * vol.shape[axis + 1] // 4
    return (
        vol[0].select(axis, idx_mid).cpu().numpy(),
        vol[0].select(axis, idx_q1).cpu().numpy(),
        vol[0].select(axis, idx_q3).cpu().numpy(),
    )

# Compute consistent vmin/vmax across original and modified for fair comparison
all_recon = torch.cat([recon_orig_from_codes, recon_modified], dim=0)
vmin_recon = all_recon.min().item()
vmax_recon = all_recon.max().item()

fig, axes = plt.subplots(3, 4, figsize=(20, 15))
titles = ["Original input", "Decoded (original codes)", "Decoded (modified codes)", "Difference (amplified)"]
volumes = [image, recon_orig_from_codes, recon_modified, diff]
slice_labels = ["mid", "Q1", "Q3"]

for i, (vol, title) in enumerate(zip(volumes, titles)):
    slices = get_mid_slice(vol[0], SLICE_AXIS)
    for j in range(3):
        if "difference" in title.lower():
            # Amplify diff: use vmax=diff.max() so the colormap fills the actual range
            diff_max = diff.max().item()
            if diff_max < 1e-8:
                diff_max = 1.0  # avoid zero range
            im = axes[j, i].imshow(slices[j], cmap="hot", origin="lower", vmin=0, vmax=diff_max)
            if j == 0:
                plt.colorbar(im, ax=axes[j, i], fraction=0.046, pad=0.04)
        elif "input" in title.lower():
            axes[j, i].imshow(slices[j], cmap="gray", origin="lower")
        else:
            # Use consistent scale for original vs modified so differences are visible
            axes[j, i].imshow(slices[j], cmap="gray", origin="lower", vmin=vmin_recon, vmax=vmax_recon)
        axes[j, i].set_title(f"{title} ({slice_labels[j]})")
        axes[j, i].axis("off")

subject_name = subjects[SUBJECT_IDX] if SUBJECT_IDX < len(subjects) else f"#{SUBJECT_IDX}"
fig.suptitle(
    f"Subject {subject_name} — Level {TARGET_LEVEL}, code {OLD_CODE} → {NEW_CODE} "
    f"({n_replaced} voxels replaced)\n"
    f"Embedding L2 dist={l2_dist:.4f}, cos_sim={cos_sim:.4f}, max_diff={diff.max().item():.6f}",
    fontsize=13,
)
plt.tight_layout()
plt.show()

## 11. Feature Map Extraction & Analysis

Visualize encoder feature maps at each level to understand what spatial patterns
the model learns. Includes per-channel activation maps, class-wise activation
statistics, and top-activated channel analysis.

In [ ]:
# Extract full spatial encoder feature maps for a few subjects per class
# NOTE: model.forward() sets encoder_outputs[l] = None after consuming each level,
# so we run the encoder stack directly to get the spatial maps.
N_PER_CLASS = 3  # subjects per class to visualize
SLICE_AXIS_FM = 2  # 0=sagittal, 1=coronal, 2=axial

# Pick subjects: first N_PER_CLASS from each class
selected_indices = []
for cls_idx in range(len(CLASS_NAMES)):
    cls_mask = np.where(labels == cls_idx)[0]
    selected_indices.extend(cls_mask[:N_PER_CLASS].tolist())

# Extract spatial feature maps by running encoders directly
feature_maps = {idx: [] for idx in selected_indices}  # idx -> list of (C, D, H, W) per level

with torch.no_grad():
    for idx in tqdm(selected_indices, desc="Extracting feature maps"):
        sample = dataset[idx]
        image = sample["image"].unsqueeze(0).to(DEVICE)

        # Run encoder stack manually (mirrors the encoder loop in model.forward)
        encoder_outputs = []
        for enc in model.encoders:
            if len(encoder_outputs):
                encoder_outputs.append(enc(encoder_outputs[-1]))
            else:
                encoder_outputs.append(enc(image))

        for lvl in range(nb_levels):
            feature_maps[idx].append(encoder_outputs[lvl][0].cpu())  # (C, D, H, W)

print(f"Extracted spatial feature maps for {len(selected_indices)} subjects")
for lvl in range(nb_levels):
    sample_feat = feature_maps[selected_indices[0]][lvl]
    print(f"  Level {lvl}: {tuple(sample_feat.shape)}")

### 11a. Per-Channel Activation Maps

Show the top-K most activated channels (by mean activation) for one subject per class at each encoder level.

In [ ]:
TOP_K_CHANNELS = 8  # number of top channels to display

for lvl in range(nb_levels):
    # Use subjects we already extracted feature maps for (from the extraction cell above)
    # Pick the first extracted subject per class
    show_indices = []
    for c in range(len(CLASS_NAMES)):
        for idx in selected_indices:
            if labels[idx] == c:
                show_indices.append(idx)
                break

    if not show_indices:
        print(f"Level {lvl}: no feature maps available — run the extraction cell first")
        continue

    n_rows = len(show_indices)
    fig, axes = plt.subplots(
        n_rows, TOP_K_CHANNELS,
        figsize=(2.5 * TOP_K_CHANNELS, 3 * n_rows),
    )
    if n_rows == 1:
        axes = axes[np.newaxis, :]  # ensure 2D indexing
    fig.suptitle(f"Level {lvl} — Top-{TOP_K_CHANNELS} Activated Channels (mid-axial slice)", fontsize=14)

    for row, idx in enumerate(show_indices):
        feat = feature_maps[idx][lvl]  # (C, D, H, W)

        # Mean activation per channel → pick the most activated ones
        chan_means = feat.mean(dim=(1, 2, 3))  # (C,)
        top_chans = torch.argsort(chan_means, descending=True)[:TOP_K_CHANNELS]

        mid = feat.shape[1 + SLICE_AXIS_FM] // 2  # mid slice along chosen axis
        for col, ch in enumerate(top_chans):
            slc = feat[ch].select(SLICE_AXIS_FM, mid).numpy()
            ax = axes[row, col]
            im = ax.imshow(slc, cmap="inferno", origin="lower")
            ax.set_title(f"ch{ch.item()} ({chan_means[ch]:.2f})", fontsize=9)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(CLASS_NAMES[labels[idx]], fontsize=12,
                              rotation=0, labelpad=30)

    plt.tight_layout()
    plt.show()

### 11b. Channel Activation Distributions by Class

Compare the distribution of mean channel activations across diagnostic classes.
Channels where distributions diverge are potentially class-discriminative.

In [ ]:
# Compute per-channel mean activation for all subjects (using the pooled features from Section 3)
# all_pooled[lvl] has shape (N, C) — each entry is the global-average-pooled encoder output

from scipy.stats import kruskal

for lvl in range(nb_levels):
    X = all_pooled[lvl]  # (N, C)
    n_channels = X.shape[1]

    # Kruskal-Wallis test per channel: non-parametric test for class differences
    kw_stats = np.zeros(n_channels)
    kw_pvals = np.zeros(n_channels)
    for ch in range(n_channels):
        groups = [X[labels == c, ch] for c in range(len(CLASS_NAMES))]
        if all(len(g) > 1 for g in groups):
            stat, pval = kruskal(*groups)
            kw_stats[ch] = stat
            kw_pvals[ch] = pval

    # Top discriminative channels by Kruskal-Wallis statistic
    top_disc = np.argsort(kw_stats)[::-1][:16]

    fig, axes = plt.subplots(4, 4, figsize=(16, 12))
    fig.suptitle(
        f"Level {lvl} — Top-16 Class-Discriminative Channels (Kruskal-Wallis)",
        fontsize=14,
    )
    for ax_idx, ch in enumerate(top_disc):
        ax = axes[ax_idx // 4, ax_idx % 4]
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            vals = X[labels == cls_idx, ch]
            ax.hist(vals, bins=25, alpha=0.5, label=cls_name,
                    color=CLASS_COLORS[cls_name], density=True)
        ax.set_title(f"ch{ch} (H={kw_stats[ch]:.1f}, p={kw_pvals[ch]:.1e})", fontsize=9)
        if ax_idx == 0:
            ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # Summary table
    print(f"Level {lvl} — Top-10 most discriminative channels:")
    print(f"  {'Channel':>8s}  {'KW stat':>10s}  {'p-value':>12s}  ", end="")
    print("  ".join(f"{'Mean ' + c:>10s}" for c in CLASS_NAMES))
    for ch in top_disc[:10]:
        means = [X[labels == c, ch].mean() for c in range(len(CLASS_NAMES))]
        print(f"  {ch:8d}  {kw_stats[ch]:10.2f}  {kw_pvals[ch]:12.2e}  ", end="")
        print("  ".join(f"{m:10.4f}" for m in means))
    print()

### 11c. Spatial Activation Difference Maps

For the most discriminative channels, show the mean spatial activation map per class
and the AD-vs-CN difference map to highlight regions where feature representations diverge.

In [ ]:
# Accumulate spatial feature maps across all subjects for class-mean maps
# We run encoders directly (model.forward nullifies encoder_outputs after use)

TOP_DISC_CHANNELS = 4  # number of discriminative channels to visualize spatially

for lvl in range(nb_levels):
    # Identify top discriminative channels from pooled features (reuse kw_stats logic)
    X = all_pooled[lvl]
    n_channels = X.shape[1]
    kw_stats_lvl = np.zeros(n_channels)
    for ch in range(n_channels):
        groups = [X[labels == c, ch] for c in range(len(CLASS_NAMES))]
        if all(len(g) > 1 for g in groups):
            stat, _ = kruskal(*groups)
            kw_stats_lvl[ch] = stat
    # .copy() is critical: [::-1] creates a view with negative strides,
    # which PyTorch cannot use as a tensor index
    top_chans = np.argsort(kw_stats_lvl)[::-1][:TOP_DISC_CHANNELS].copy()

    # Accumulate mean spatial maps per class for the top channels
    class_sum = {c: None for c in range(len(CLASS_NAMES))}
    class_count = {c: 0 for c in range(len(CLASS_NAMES))}

    with torch.no_grad():
        for batch_start in tqdm(range(0, len(dataset), BATCH_SIZE),
                                desc=f"Level {lvl} spatial maps"):
            batch_end = min(batch_start + BATCH_SIZE, len(dataset))
            batch_images = torch.stack(
                [dataset[i]["image"] for i in range(batch_start, batch_end)]
            ).to(DEVICE)

            # Run encoder stack directly to get spatial feature maps
            encoder_outputs = []
            for enc in model.encoders:
                if len(encoder_outputs):
                    encoder_outputs.append(enc(encoder_outputs[-1]))
                else:
                    encoder_outputs.append(enc(batch_images))

            feat = encoder_outputs[lvl]  # (B, C, D, H, W)

            # Only keep the top discriminative channels
            feat_sel = feat[:, top_chans].cpu().float()  # (B, TOP_DISC_CHANNELS, D, H, W)

            for b in range(feat_sel.shape[0]):
                cls = labels[batch_start + b]
                if class_sum[cls] is None:
                    class_sum[cls] = torch.zeros_like(feat_sel[b])
                class_sum[cls] += feat_sel[b]
                class_count[cls] += 1

    # Compute class means (only for classes that have subjects)
    class_means = {}
    for c in range(len(CLASS_NAMES)):
        if class_count[c] > 0:
            class_means[c] = (class_sum[c] / class_count[c]).numpy()

    # Skip if any class is missing
    if len(class_means) < len(CLASS_NAMES):
        missing = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if c not in class_means]
        print(f"Level {lvl}: skipping — no subjects for classes: {missing}")
        continue

    # Visualize: rows = channels, columns = classes + AD-CN difference
    n_cols = len(CLASS_NAMES) + 1  # +1 for difference map
    fig, axes = plt.subplots(
        TOP_DISC_CHANNELS, n_cols,
        figsize=(4 * n_cols, 3.5 * TOP_DISC_CHANNELS),
    )
    if TOP_DISC_CHANNELS == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f"Level {lvl} — Mean Spatial Activations (mid-axial slice)", fontsize=14)

    for row, ch_global in enumerate(top_chans):
        # Get mid slice for each class
        class_slices = {}
        for c in range(len(CLASS_NAMES)):
            vol = class_means[c][row]  # (D, H, W) — row indexes into selected channels
            mid = vol.shape[SLICE_AXIS_FM] // 2
            class_slices[c] = np.take(vol, mid, axis=SLICE_AXIS_FM)

        # Common colorscale across classes
        vmin = min(s.min() for s in class_slices.values())
        vmax = max(s.max() for s in class_slices.values())

        for col, c in enumerate(range(len(CLASS_NAMES))):
            ax = axes[row, col]
            im = ax.imshow(class_slices[c], cmap="inferno", origin="lower",
                           vmin=vmin, vmax=vmax)
            ax.set_title(f"{CLASS_NAMES[c]}", fontsize=10)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(f"ch{ch_global}\n(H={kw_stats_lvl[ch_global]:.1f})",
                              fontsize=10, rotation=0, labelpad=50)

        # AD - CN difference map
        ad_idx = CLASS_NAMES.index("AD")
        cn_idx = CLASS_NAMES.index("CN")
        diff_map = class_slices[ad_idx] - class_slices[cn_idx]
        ax = axes[row, -1]
        abs_max = max(abs(diff_map.min()), abs(diff_map.max()))
        if abs_max < 1e-8:
            abs_max = 1.0
        im = ax.imshow(diff_map, cmap="RdBu_r", origin="lower",
                       vmin=-abs_max, vmax=abs_max)
        ax.set_title("AD - CN", fontsize=10)
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

### 11d. Feature Map Summary Statistics Across Levels

Compare activation magnitude, sparsity, and variance across encoder levels to understand
what each level captures (fine texture vs coarse structure).

In [ ]:
# Compute summary statistics from pooled features (already extracted)
stats_data = []

for lvl in range(nb_levels):
    X = all_pooled[lvl]  # (N, C)
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        X_cls = X[mask]
        stats_data.append({
            "Level": lvl,
            "Class": cls_name,
            "Mean activation": X_cls.mean(),
            "Std activation": X_cls.std(),
            "Sparsity (% near-zero)": (np.abs(X_cls) < 0.01).mean() * 100,
            "Max activation": X_cls.max(),
            "Active channels (>0.1)": (np.abs(X_cls).mean(axis=0) > 0.1).sum(),
        })

stats_df = pd.DataFrame(stats_data)
print(stats_df.to_string(index=False))

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, metric in zip(axes, ["Mean activation", "Std activation", "Sparsity (% near-zero)"]):
    for cls_name in CLASS_NAMES:
        subset = stats_df[stats_df["Class"] == cls_name]
        ax.plot(subset["Level"], subset[metric], "o-",
                color=CLASS_COLORS[cls_name], label=cls_name, linewidth=2, markersize=8)
    ax.set_xlabel("Encoder Level")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xticks(range(nb_levels))
    ax.legend()

plt.suptitle("Feature Map Statistics Across Encoder Levels", fontsize=14)
plt.tight_layout()
plt.show()

## 12. Codebook Distributions Across Diagnosis Codes

Detailed distributional analysis of how codebook usage differs between diagnostic groups (AD, CN, MCI).

- **12a**: Per-class mean codebook distributions with JS-divergence between every class pair
- **12b**: Per-code statistical testing (Kruskal-Wallis) to find codes with significantly different usage across groups
- **12c**: Violin plots for the top discriminative codes showing the full per-subject distribution by class
- **12d**: Class-pair divergence profiles — which codebook entries contribute most to each pairwise divergence

### 12a. Jensen-Shannon Divergence Between Class Codebook Distributions

For each level, compute the mean codebook usage distribution per class, then measure the JS-divergence between every class pair. JSD is bounded [0, 1] (using log base 2) — higher values indicate the two classes use the codebook more differently.

In [ ]:
from scipy.spatial.distance import jensenshannon
from itertools import combinations

for lvl in range(nb_levels):
    # Mean codebook distribution per class (add small epsilon for numerical stability)
    eps = 1e-12
    class_dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        mean_hist = all_histograms[lvl][mask].mean(axis=0) + eps
        mean_hist /= mean_hist.sum()  # re-normalise
        class_dists[cls_name] = mean_hist

    # JSD matrix
    n_cls = len(CLASS_NAMES)
    jsd_matrix = np.zeros((n_cls, n_cls))
    for (i, name_i), (j, name_j) in combinations(enumerate(CLASS_NAMES), 2):
        jsd = jensenshannon(class_dists[name_i], class_dists[name_j], base=2) ** 2
        jsd_matrix[i, j] = jsd
        jsd_matrix[j, i] = jsd

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Level {lvl} — Class Codebook Distributions & JSD", fontsize=14)

    # Left: overlaid distributions
    ax = axes[0]
    for cls_name, dist in class_dists.items():
        ax.plot(dist, alpha=0.7, label=cls_name, color=CLASS_COLORS[cls_name], linewidth=1.2)
    ax.set_xlabel("Codebook entry")
    ax.set_ylabel("Mean frequency")
    ax.set_title("Mean codebook distribution per class")
    ax.legend()

    # Right: JSD heatmap
    ax = axes[1]
    im = ax.imshow(jsd_matrix, cmap="YlOrRd", vmin=0)
    ax.set_xticks(range(n_cls))
    ax.set_yticks(range(n_cls))
    ax.set_xticklabels(CLASS_NAMES)
    ax.set_yticklabels(CLASS_NAMES)
    for i in range(n_cls):
        for j in range(n_cls):
            ax.text(j, i, f"{jsd_matrix[i, j]:.4f}", ha="center", va="center",
                    color="white" if jsd_matrix[i, j] > jsd_matrix.max() * 0.6 else "black")
    ax.set_title("Jensen-Shannon Divergence (squared)")
    fig.colorbar(im, ax=ax, shrink=0.8)

    plt.tight_layout()
    plt.show()

    # Print summary
    for (i, ni), (j, nj) in combinations(enumerate(CLASS_NAMES), 2):
        print(f"  Level {lvl} JSD²({ni}, {nj}) = {jsd_matrix[i, j]:.6f}")

### 12b. Per-Code Statistical Testing (Kruskal-Wallis)

For each codebook entry, test whether its frequency distribution differs significantly across the three diagnostic groups using the non-parametric Kruskal-Wallis H-test (no normality assumption). Results are corrected for multiple comparisons (Benjamini-Hochberg FDR).

In [ ]:
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

TOP_K_CODES = 20  # codes to highlight per level

kw_results = {}  # lvl -> DataFrame of results

for lvl in range(nb_levels):
    h_stats, p_vals, code_indices = [], [], []

    for code_idx in range(nb_entries):
        groups = [all_histograms[lvl][labels == c, code_idx] for c in range(len(CLASS_NAMES))]
        # Skip codes with no variance — kruskal requires at least some
        # variation across all samples combined
        all_vals = np.concatenate(groups)
        if all_vals.std() == 0:
            continue
        stat, p = kruskal(*groups)
        h_stats.append(stat)
        p_vals.append(p)
        code_indices.append(code_idx)

    # FDR correction
    reject, p_adj, _, _ = multipletests(p_vals, method="fdr_bh", alpha=0.05)

    df = pd.DataFrame({
        "code": code_indices,
        "H_stat": h_stats,
        "p_value": p_vals,
        "p_adj": p_adj,
        "significant": reject,
    }).sort_values("H_stat", ascending=False).reset_index(drop=True)

    # Add per-class mean frequencies for context
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        df[f"mean_{cls_name}"] = df["code"].apply(lambda c: all_histograms[lvl][mask, c].mean())

    kw_results[lvl] = df

    n_sig = df["significant"].sum()
    n_tested = len(df)
    print(f"Level {lvl}: {n_sig}/{n_tested} codes significant (FDR < 0.05)")

    # Plot: H-statistic across all codes (highlight significant)
    fig, axes = plt.subplots(1, 2, figsize=(18, 5))
    fig.suptitle(f"Level {lvl} — Kruskal-Wallis Test per Codebook Entry", fontsize=14)

    ax = axes[0]
    colors = ["#e74c3c" if s else "#95a5a6" for s in df["significant"]]
    ax.bar(range(len(df)), df["H_stat"], color=colors, width=1.0)
    ax.set_xlabel("Code (sorted by H-statistic)")
    ax.set_ylabel("Kruskal-Wallis H")
    ax.set_title(f"H-statistic (red = FDR < 0.05, {n_sig} significant)")

    # Right: volcano-style — H-stat vs -log10(p_adj)
    ax = axes[1]
    neg_log_p = -np.log10(df["p_adj"].clip(lower=1e-300))
    ax.scatter(df["H_stat"], neg_log_p, c=colors, s=12, alpha=0.7)
    ax.axhline(-np.log10(0.05), color="gray", linestyle="--", linewidth=0.8, label="FDR = 0.05")
    ax.set_xlabel("Kruskal-Wallis H")
    ax.set_ylabel("-log10(adjusted p-value)")
    ax.set_title("Volcano plot")
    ax.legend()

    plt.tight_layout()
    plt.show()

    # Print top codes
    print(f"\n  Top {TOP_K_CODES} most discriminative codes (by H-stat):")
    top = df.head(TOP_K_CODES)
    for _, row in top.iterrows():
        sig_marker = "*" if row["significant"] else " "
        means = " | ".join(f"{cls}: {row[f'mean_{cls}']:.4f}" for cls in CLASS_NAMES)
        print(f"  {sig_marker} Code {int(row['code']):>3d}  H={row['H_stat']:7.2f}  "
              f"p_adj={row['p_adj']:.2e}  [{means}]")
    print()

### 12c. Violin Plots for Top Discriminative Codes

For the top-K most discriminative codes (by Kruskal-Wallis H-stat) at each level, show violin + strip plots of the per-subject frequency distributions broken down by class. This reveals whether the difference is in central tendency, spread, or shape.

In [ ]:
TOP_K_VIOLIN = 8  # codes to plot per level

for lvl in range(nb_levels):
    df = kw_results[lvl]
    top_codes = df.head(TOP_K_VIOLIN)["code"].values.astype(int)

    n_codes = len(top_codes)
    fig, axes = plt.subplots(2, (n_codes + 1) // 2, figsize=(4 * ((n_codes + 1) // 2), 10))
    axes = axes.flatten()
    fig.suptitle(f"Level {lvl} — Top {n_codes} Discriminative Code Distributions", fontsize=14)

    for ax_idx, code_idx in enumerate(top_codes):
        ax = axes[ax_idx]

        # Build a long-form DataFrame for seaborn
        plot_data = []
        for cls_idx, cls_name in enumerate(CLASS_NAMES):
            mask = labels == cls_idx
            freqs = all_histograms[lvl][mask, code_idx]
            for f in freqs:
                plot_data.append({"Class": cls_name, "Frequency": f})
        plot_df = pd.DataFrame(plot_data)

        sns.violinplot(
            data=plot_df, x="Class", y="Frequency", ax=ax,
            palette=CLASS_COLORS, inner=None, alpha=0.3, linewidth=0.8,
            order=CLASS_NAMES,
        )
        sns.stripplot(
            data=plot_df, x="Class", y="Frequency", ax=ax,
            palette=CLASS_COLORS, size=2, alpha=0.4, jitter=True,
            order=CLASS_NAMES,
        )

        row = df[df["code"] == code_idx].iloc[0]
        sig = "***" if row["p_adj"] < 0.001 else "**" if row["p_adj"] < 0.01 else "*" if row["p_adj"] < 0.05 else "ns"
        ax.set_title(f"Code {code_idx} ({sig})\nH={row['H_stat']:.1f}", fontsize=10)
        ax.set_xlabel("")

    # Hide unused axes
    for ax_idx in range(n_codes, len(axes)):
        axes[ax_idx].set_visible(False)

    plt.tight_layout()
    plt.show()

### 12d. Class-Pair Divergence Profiles

For each pair of diagnostic classes, compute the per-entry contribution to the overall JS-divergence. This shows *which specific codebook entries* drive the distributional difference between any two classes — useful for identifying disease-specific encoding patterns.

In [ ]:
def per_entry_jsd(p, q):
    """Compute per-bin contribution to JS-divergence (sums to JSD)."""
    eps = 1e-12
    p = np.asarray(p, dtype=np.float64) + eps
    q = np.asarray(q, dtype=np.float64) + eps
    p /= p.sum()
    q /= q.sum()
    m = 0.5 * (p + q)
    # Per-entry: 0.5 * [p_i * log(p_i/m_i) + q_i * log(q_i/m_i)]
    contrib = 0.5 * (p * np.log2(p / m) + q * np.log2(q / m))
    return contrib  # (nb_entries,), sums to JSD²


class_pairs = list(combinations(range(len(CLASS_NAMES)), 2))
pair_names = [(CLASS_NAMES[i], CLASS_NAMES[j]) for i, j in class_pairs]
pair_colors = ["#8e44ad", "#e67e22", "#16a085"]  # one color per pair

TOP_N_ENTRIES = 10  # top entries to annotate per pair

for lvl in range(nb_levels):
    eps = 1e-12

    # Mean distributions per class
    dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        d = all_histograms[lvl][labels == cls_idx].mean(axis=0) + eps
        dists[cls_idx] = d / d.sum()

    fig, axes = plt.subplots(len(class_pairs), 1, figsize=(16, 4.5 * len(class_pairs)),
                              sharex=True)
    if len(class_pairs) == 1:
        axes = [axes]
    fig.suptitle(f"Level {lvl} — Per-Entry JSD Contribution by Class Pair", fontsize=14, y=1.01)

    for ax, (ci, cj), (ni, nj), col in zip(axes, class_pairs, pair_names, pair_colors):
        contrib = per_entry_jsd(dists[ci], dists[cj])

        ax.bar(range(nb_entries), contrib, color=col, alpha=0.7, width=1.0)
        ax.set_ylabel("JSD contribution")
        ax.set_title(f"{ni} vs {nj}  (total JSD² = {contrib.sum():.6f})")

        # Annotate top entries
        top_idx = np.argsort(contrib)[-TOP_N_ENTRIES:][::-1]
        for rank, idx in enumerate(top_idx):
            if rank < 5:  # annotate top 5 only to avoid clutter
                ax.annotate(
                    f"{idx}", (idx, contrib[idx]),
                    textcoords="offset points", xytext=(0, 6),
                    fontsize=7, ha="center", color=col,
                )

    axes[-1].set_xlabel("Codebook entry")
    plt.tight_layout()
    plt.show()

    # Summary table: top entries per pair
    print(f"Level {lvl} — Top {TOP_N_ENTRIES} entries driving each class-pair divergence:")
    for (ci, cj), (ni, nj) in zip(class_pairs, pair_names):
        contrib = per_entry_jsd(dists[ci], dists[cj])
        top_idx = np.argsort(contrib)[-TOP_N_ENTRIES:][::-1]
        entries_str = ", ".join(
            f"{idx}({contrib[idx]:.5f})" for idx in top_idx
        )
        print(f"  {ni} vs {nj}: {entries_str}")
    print()

### 12e. Spatial Maps of Top JSD-Driving Codebook Entries

For each class pair, identify the codes that contribute most to the Jensen-Shannon divergence, then show the **mean spatial occupancy** of those codes in each diagnostic group. For a given code, the occupancy at each voxel is the fraction of subjects in that group where the voxel was assigned to that code. Differences between groups reveal where in the brain the model encodes disease-relevant information.

In [ ]:
# ── Spatial occupancy maps of top JSD-driving codes per class pair ──

TOP_JSD_CODES = 6   # number of top codes to visualise per pair
N_SLICES = 5         # axial slices to show

for lvl in range(nb_levels):
    indices = spatial_indices[lvl]  # (N, D, H, W)
    eps = 1e-12

    # Mean codebook distributions per class (for JSD computation)
    dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        d = all_histograms[lvl][labels == cls_idx].mean(axis=0) + eps
        dists[cls_idx] = d / d.sum()

    for (ci, cj), (ni, nj) in zip(class_pairs, pair_names):
        contrib = per_entry_jsd(dists[ci], dists[cj])
        top_codes = np.argsort(contrib)[-TOP_JSD_CODES:][::-1]

        D = indices.shape[1]
        slice_idxs = np.linspace(0, D - 1, N_SLICES, dtype=int)

        # One figure per code: rows = classes, cols = slices
        # Plus a difference row for the two classes in the pair
        n_rows = len(CLASS_NAMES) + 1  # all classes + difference map
        for code_idx in top_codes:
            fig, axes = plt.subplots(
                n_rows, N_SLICES,
                figsize=(3 * N_SLICES, 3 * n_rows),
            )
            fig.suptitle(
                f"Level {lvl}, Code {code_idx} — JSD contrib = {contrib[code_idx]:.5f}\n"
                f"(pair: {ni} vs {nj})",
                fontsize=12,
            )

            # Compute mean occupancy per class: fraction of subjects where
            # each voxel is assigned to this code
            occupancy = {}
            for cls_idx, cls_name in enumerate(CLASS_NAMES):
                mask = labels == cls_idx
                # (n_subj_in_class, D, H, W) == code_idx → bool → mean over subjects
                occupancy[cls_name] = (indices[mask] == code_idx).astype(np.float32).mean(axis=0)

            # Find global vmax across all classes for consistent colorscale
            vmax_occ = max(occ.max() for occ in occupancy.values())
            vmax_occ = max(vmax_occ, 1e-6)

            # Plot each class
            for row_idx, cls_name in enumerate(CLASS_NAMES):
                for col_idx, s in enumerate(slice_idxs):
                    ax = axes[row_idx, col_idx]
                    im = ax.imshow(
                        occupancy[cls_name][s], cmap="hot",
                        vmin=0, vmax=vmax_occ, origin="lower",
                    )
                    ax.axis("off")
                    if col_idx == 0:
                        ax.set_ylabel(cls_name, fontsize=11, rotation=0, labelpad=40, va="center")
                    if row_idx == 0:
                        ax.set_title(f"Slice {s}", fontsize=9)

            # Difference row: class_i - class_j
            diff_map = occupancy[ni] - occupancy[nj]
            vmax_diff = max(abs(diff_map.min()), abs(diff_map.max()), 1e-6)
            for col_idx, s in enumerate(slice_idxs):
                ax = axes[n_rows - 1, col_idx]
                im_diff = ax.imshow(
                    diff_map[s], cmap="RdBu_r",
                    vmin=-vmax_diff, vmax=vmax_diff, origin="lower",
                )
                ax.axis("off")
                if col_idx == 0:
                    ax.set_ylabel(f"{ni}−{nj}", fontsize=11, rotation=0, labelpad=40, va="center")

            # Colorbars
            fig.colorbar(im, ax=axes[:len(CLASS_NAMES), :].tolist(), shrink=0.6,
                        label="Occupancy (fraction of subjects)", pad=0.02)
            fig.colorbar(im_diff, ax=axes[n_rows - 1, :].tolist(), shrink=0.6,
                        label=f"Δ occupancy ({ni} − {nj})", pad=0.02)

            plt.tight_layout()
            plt.show()

## 14. Codebook Distributions by Demographic Factors

Load the merged demographic CSV and analyse how codebook usage varies across **gender**, **age group**, **education**, and **race** — both within and across diagnostic groups. For each factor we compute:
- JSD matrices between subgroups
- Per-code Kruskal-Wallis tests (FDR corrected)
- Interaction with diagnosis (JSD stratified by Group)

In [ ]:
# ── Load merged demographics and join to notebook subjects ──

MERGED_CSV = "../merged_data.csv"  # columns: Subject, Group, PTGENDER, AGE, PTEDUCAT, PTRACCAT

demo_df = pd.read_csv(MERGED_CSV)

# Deduplicate: keep first row per subject (multiple visits in ADNI)
demo_df = demo_df.drop_duplicates(subset="Subject", keep="first")
print(f"Demographic CSV: {len(demo_df)} unique subjects")

# Build a DataFrame for our notebook subjects (preserving order to match
# all_histograms / labels arrays — these may be subsampled via MAX_SUBJECTS)
subj_df = pd.DataFrame({"Subject": subjects, "_idx": np.arange(len(subjects))})
subj_df = subj_df.merge(demo_df, on="Subject", how="left")
subj_df = subj_df.sort_values("_idx").reset_index(drop=True)

assert len(subj_df) == len(subjects), (
    f"Merge changed row count: {len(subj_df)} vs {len(subjects)} — "
    "check for duplicate Subject IDs in merged_data.csv"
)

# Report merge quality
n_matched = subj_df["AGE"].notna().sum()
print(f"Matched {n_matched}/{len(subjects)} subjects to demographic data "
      f"({len(subjects) - n_matched} missing)")

# ── Create categorical groupings ──

# Gender: 1 = Male, 2 = Female (ADNI coding)
GENDER_MAP = {1.0: "Male", 2.0: "Female"}
subj_df["Gender"] = subj_df["PTGENDER"].map(GENDER_MAP)

# Age: bin into decades
age_bins = [0, 65, 75, 85, 200]
age_labels = ["<65", "65-74", "75-84", "85+"]
subj_df["AgeGroup"] = pd.cut(subj_df["AGE"], bins=age_bins, labels=age_labels, right=False)

# Education: low / medium / high
edu_bins = [0, 13, 17, 100]
edu_labels = ["<=12 yrs", "13-16 yrs", "17+ yrs"]
subj_df["Education"] = pd.cut(subj_df["PTEDUCAT"], bins=edu_bins, labels=edu_labels, right=True)

# Race (keep top categories, group rare ones)
RACE_MAP = {
    "1": "Native American", "2": "Asian", "3": "Pacific Islander",
    "4": "Black", "5": "White", "6": "More than one", "7": "Unknown",
}
subj_df["Race"] = subj_df["PTRACCAT"].astype(str).map(RACE_MAP).fillna("Other")

# Summary
for col in ["Gender", "AgeGroup", "Education", "Race"]:
    print(f"\n{col}:")
    print(subj_df[col].value_counts().to_string())

### 14a. JSD Between Demographic Subgroups

For each demographic factor and codebook level, compute the pairwise Jensen-Shannon divergence between the mean codebook distributions of each subgroup. This reveals which demographic splits create the most distinct encoding patterns.

In [ ]:
# ── JSD matrices for each demographic factor ──

from scipy.spatial.distance import jensenshannon
from itertools import combinations

DEMO_FACTORS = {
    "Gender": "Gender",
    "Age Group": "AgeGroup",
    "Education": "Education",
    "Race": "Race",
}

# Store results for later use
demo_jsd_results = {}  # (factor_name, lvl) -> (group_names, jsd_matrix)

for factor_name, col in DEMO_FACTORS.items():
    # Drop subjects with missing demographic data for this factor
    valid_mask = subj_df[col].notna().values
    group_vals = subj_df.loc[valid_mask, col].values
    unique_groups = sorted([g for g in subj_df[col].dropna().unique()], key=str)

    # Skip factors with only one group
    if len(unique_groups) < 2:
        print(f"Skipping {factor_name}: only {len(unique_groups)} group(s)")
        continue

    for lvl in range(nb_levels):
        hists = all_histograms[lvl][valid_mask]  # (N_valid, nb_entries)
        eps = 1e-12

        # Mean codebook distribution per subgroup
        group_dists = {}
        group_counts = {}
        for g in unique_groups:
            mask = group_vals == g
            if mask.sum() < 5:
                continue  # skip very small groups
            mean_hist = hists[mask].mean(axis=0) + eps
            mean_hist /= mean_hist.sum()
            group_dists[str(g)] = mean_hist
            group_counts[str(g)] = int(mask.sum())

        gnames = list(group_dists.keys())
        n_g = len(gnames)
        if n_g < 2:
            continue

        # Pairwise JSD
        jsd_matrix = np.zeros((n_g, n_g))
        for (i, gi), (j, gj) in combinations(enumerate(gnames), 2):
            jsd = jensenshannon(group_dists[gi], group_dists[gj], base=2) ** 2
            jsd_matrix[i, j] = jsd
            jsd_matrix[j, i] = jsd

        demo_jsd_results[(factor_name, lvl)] = (gnames, jsd_matrix, group_counts)

        # Plot
        fig, ax = plt.subplots(figsize=(max(4, n_g * 1.2), max(3, n_g * 1.0)))
        tick_labels = [f"{g}\n(n={group_counts[g]})" for g in gnames]
        im = ax.imshow(jsd_matrix, cmap="YlOrRd", vmin=0)
        ax.set_xticks(range(n_g))
        ax.set_xticklabels(tick_labels, rotation=45, ha="right", fontsize=9)
        ax.set_yticks(range(n_g))
        ax.set_yticklabels(tick_labels, fontsize=9)

        # Annotate cells
        for i in range(n_g):
            for j in range(n_g):
                if i != j:
                    ax.text(j, i, f"{jsd_matrix[i, j]:.4f}", ha="center", va="center", fontsize=8)

        fig.colorbar(im, ax=ax, shrink=0.8, label="JSD (bits²)")
        ax.set_title(f"Level {lvl} — {factor_name}: Pairwise JSD of Codebook Distributions", fontsize=11)
        plt.tight_layout()
        plt.show()

        # Print max divergence pair
        idx_flat = np.argmax(jsd_matrix)
        i_max, j_max = np.unravel_index(idx_flat, jsd_matrix.shape)
        print(f"  Level {lvl}, {factor_name}: max JSD = {jsd_matrix[i_max, j_max]:.5f} "
              f"between {gnames[i_max]} and {gnames[j_max]}")

### 14b. Per-Code Kruskal-Wallis Tests by Demographic Factor

For each demographic factor, test which individual codebook entries differ significantly across subgroups (FDR corrected). This complements the global JSD by pinpointing *which codes* drive the divergence.

In [ ]:
# ── Per-code Kruskal-Wallis tests for each demographic factor ──

from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

TOP_K_DEMO = 15  # top codes to show per factor per level

demo_kw_results = {}  # (factor_name, lvl) -> DataFrame

for factor_name, col in DEMO_FACTORS.items():
    valid_mask = subj_df[col].notna().values
    group_vals = subj_df.loc[valid_mask, col].values
    unique_groups = sorted([str(g) for g in subj_df[col].dropna().unique()])

    # Filter to groups with enough subjects
    group_sizes = {str(g): (group_vals == g).sum() if not isinstance(g, str)
                   else (group_vals.astype(str) == g).sum()
                   for g in subj_df[col].dropna().unique()}
    keep_groups = [g for g, n in group_sizes.items() if n >= 5]
    if len(keep_groups) < 2:
        continue

    for lvl in range(nb_levels):
        hists = all_histograms[lvl][valid_mask]
        h_stats, p_vals, code_indices = [], [], []

        for code_idx in range(nb_entries):
            groups = []
            for g in keep_groups:
                mask = group_vals.astype(str) == g
                vals = hists[mask, code_idx]
                if len(vals) > 1:
                    groups.append(vals)
            if len(groups) < 2:
                continue
            all_vals = np.concatenate(groups)
            if all_vals.std() == 0:
                continue
            stat, p = kruskal(*groups)
            h_stats.append(stat)
            p_vals.append(p)
            code_indices.append(code_idx)

        if len(p_vals) == 0:
            continue

        reject, p_adj, _, _ = multipletests(p_vals, method="fdr_bh", alpha=0.05)

        df_kw = pd.DataFrame({
            "code": code_indices,
            "H_stat": h_stats,
            "p_value": p_vals,
            "p_adj": p_adj,
            "significant": reject,
        }).sort_values("H_stat", ascending=False).reset_index(drop=True)

        demo_kw_results[(factor_name, lvl)] = df_kw

        n_sig = df_kw["significant"].sum()
        print(f"Level {lvl}, {factor_name}: {n_sig}/{len(df_kw)} codes significant (FDR < 0.05)")

        # Bar plot of top codes
        top = df_kw.head(TOP_K_DEMO)
        fig, ax = plt.subplots(figsize=(10, 4))
        colors = ["#e74c3c" if s else "#95a5a6" for s in top["significant"]]
        ax.bar(range(len(top)), top["H_stat"].values, color=colors, edgecolor="white")
        ax.set_xticks(range(len(top)))
        ax.set_xticklabels([f"{int(c)}" for c in top["code"]], fontsize=8)
        ax.set_xlabel("Codebook entry")
        ax.set_ylabel("Kruskal-Wallis H statistic")
        ax.set_title(f"Level {lvl} — Top {TOP_K_DEMO} codes by {factor_name} "
                     f"({n_sig} significant, red = FDR < 0.05)")
        plt.tight_layout()
        plt.show()

### 14c. Demographic JSD Stratified by Diagnosis

The key question: are demographic codebook differences *confounded* with diagnosis, or do they exist independently? For each diagnostic group (AD, CN, MCI), compute the JSD across demographic subgroups *within* that diagnosis. If gender/age/etc. drive codebook differences even within a single diagnostic group, the model may be encoding demographic information rather than (or in addition to) disease-related features.

In [ ]:
# ── JSD by demographic factor, stratified within each diagnosis ──

for factor_name, col in DEMO_FACTORS.items():
    valid_mask = subj_df[col].notna().values
    group_vals = subj_df.loc[valid_mask, col].astype(str).values
    diag_vals = subj_df.loc[valid_mask, "Group"].values

    keep_groups = [str(g) for g in subj_df[col].dropna().unique()
                   if (subj_df[col].astype(str) == str(g)).sum() >= 5]
    if len(keep_groups) < 2:
        continue

    for lvl in range(nb_levels):
        hists = all_histograms[lvl][valid_mask]
        eps = 1e-12

        # One row of JSD values per diagnosis
        diag_names = sorted([d for d in subj_df["Group"].dropna().unique()])
        results_rows = []

        for diag in diag_names:
            diag_mask = diag_vals == diag
            sub_groups = group_vals[diag_mask]
            sub_hists = hists[diag_mask]

            # Mean distribution per demographic subgroup within this diagnosis
            dists = {}
            counts = {}
            for g in keep_groups:
                g_mask = sub_groups == g
                if g_mask.sum() < 3:
                    continue
                h = sub_hists[g_mask].mean(axis=0) + eps
                h /= h.sum()
                dists[g] = h
                counts[g] = int(g_mask.sum())

            if len(dists) < 2:
                continue

            # All pairwise JSD within this diagnosis
            pairs = list(combinations(dists.keys(), 2))
            for g1, g2 in pairs:
                jsd = jensenshannon(dists[g1], dists[g2], base=2) ** 2
                results_rows.append({
                    "Diagnosis": diag,
                    "Group1": f"{g1} (n={counts[g1]})",
                    "Group2": f"{g2} (n={counts[g2]})",
                    "JSD": jsd,
                })

        if not results_rows:
            continue

        res_df = pd.DataFrame(results_rows)

        # Plot grouped bar chart
        fig, ax = plt.subplots(figsize=(max(8, len(res_df) * 0.8), 4))
        diag_colors = {"AD": "#e74c3c", "CN": "#2ecc71", "MCI": "#3498db"}

        x_labels = []
        x_vals = []
        colors = []
        for idx, row in res_df.iterrows():
            x_labels.append(f"{row['Group1']}\nvs\n{row['Group2']}")
            x_vals.append(row["JSD"])
            colors.append(diag_colors.get(row["Diagnosis"], "#888"))

        bars = ax.bar(range(len(x_vals)), x_vals, color=colors, edgecolor="white", alpha=0.85)
        ax.set_xticks(range(len(x_vals)))
        ax.set_xticklabels(x_labels, fontsize=7, rotation=45, ha="right")
        ax.set_ylabel("JSD (bits²)")
        ax.set_title(f"Level {lvl} — {factor_name}: JSD Within Each Diagnosis", fontsize=11)

        # Legend
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=diag_colors[d], label=d) for d in diag_names
                          if d in res_df["Diagnosis"].values]
        ax.legend(handles=legend_elements, loc="upper right")

        plt.tight_layout()
        plt.show()

        # Print summary table
        print(res_df.to_string(index=False, float_format="{:.5f}".format))

### 14d. Summary: Demographic vs Diagnostic JSD Comparison

Compare the magnitude of JSD from diagnosis (AD vs CN vs MCI) against demographic factors. If demographic JSD is comparable to diagnostic JSD, the codebook may be encoding confounding demographic variation.

In [ ]:
# ── Summary bar chart: max JSD per factor vs diagnosis JSD ──

for lvl in range(nb_levels):
    summary_rows = []

    # Diagnosis JSD (from section 12 — recompute to be safe)
    eps = 1e-12
    diag_dists = {}
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        mask = labels == cls_idx
        h = all_histograms[lvl][mask].mean(axis=0) + eps
        h /= h.sum()
        diag_dists[cls_name] = h

    for (i, ni), (j, nj) in combinations(enumerate(CLASS_NAMES), 2):
        jsd = jensenshannon(diag_dists[ni], diag_dists[nj], base=2) ** 2
        summary_rows.append({"Factor": "Diagnosis", "Pair": f"{ni} vs {nj}", "JSD": jsd})

    # Demographic JSD (from 14a results)
    for factor_name, col in DEMO_FACTORS.items():
        key = (factor_name, lvl)
        if key not in demo_jsd_results:
            continue
        gnames, jsd_mat, gcounts = demo_jsd_results[key]
        for (i, gi), (j, gj) in combinations(enumerate(gnames), 2):
            summary_rows.append({
                "Factor": factor_name,
                "Pair": f"{gi} vs {gj}",
                "JSD": jsd_mat[i, j],
            })

    if not summary_rows:
        continue

    sum_df = pd.DataFrame(summary_rows)

    # Plot
    fig, ax = plt.subplots(figsize=(max(10, len(sum_df) * 0.6), 5))
    factor_colors = {
        "Diagnosis": "#2c3e50",
        "Gender": "#8e44ad",
        "Age Group": "#e67e22",
        "Education": "#16a085",
        "Race": "#c0392b",
    }
    colors = [factor_colors.get(f, "#888") for f in sum_df["Factor"]]
    bars = ax.bar(range(len(sum_df)), sum_df["JSD"].values, color=colors, edgecolor="white")
    ax.set_xticks(range(len(sum_df)))
    ax.set_xticklabels(sum_df["Pair"], rotation=60, ha="right", fontsize=8)
    ax.set_ylabel("JSD (bits²)")
    ax.set_title(f"Level {lvl} — All Pairwise JSD: Diagnosis vs Demographics", fontsize=12)

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=f) for f, c in factor_colors.items()
                      if f in sum_df["Factor"].values]
    ax.legend(handles=legend_elements, loc="upper right")
    plt.tight_layout()
    plt.show()

    # Print sorted table
    print(f"\nLevel {lvl} — All pairwise JSD (sorted):")
    print(sum_df.sort_values("JSD", ascending=False).to_string(index=False, float_format="{:.5f}".format))